# Code 8.3.2: Packing documents into blocks

Appends GPT-2's end-of-text token to each surviving document, concatenates them, and cuts the stream into blocks of $`n + 1 = 9`$ tokens. It reuses the definitions of Code 8.3.1.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


### Setup from Code 8.3.1

Included so the next cell can reuse its definitions and variables.


In [ ]:
import hashlib, re
import torch
import tiktoken

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token                                     # <|endoftext|>, id 50256
page = ("The museum is open from nine in the morning until five in the evening, "
        "and admission is free for children under twelve and for students with a valid card.")
docs = [page,
        "Pretraining data is measured in tokens.",
        page.replace(" ", "  "),                          # duplicate up to whitespace
        page.replace("five", "six"),                      # near duplicate: one word changed
        "A short one."]

# Exact deduplication: hash a normalized copy of each document
def normalize(s):
    return re.sub(r"\s+", " ", s.lower()).strip()
seen, unique = set(), []
for doc in docs:
    key = hashlib.sha256(normalize(doc).encode()).hexdigest()
    if key not in seen:
        seen.add(key); unique.append(doc)
print(len(docs), "->", len(unique), "documents after exact dedup")

# Near-duplicate detection: Jaccard similarity of word 3-gram sets, estimated with MinHash
def shingles(s, k=3):
    w = re.findall(r"\w+", s.lower())
    return {" ".join(w[i:i + k]) for i in range(len(w) - k + 1)}
def minhash(sh, num_hashes=128):
    return [min(int(hashlib.md5(f"{i}:{g}".encode()).hexdigest()[:8], 16) for g in sh)
            for i in range(num_hashes)]
a, b = shingles(unique[0]), shingles(unique[2])
jaccard = len(a & b) / len(a | b)
ma, mb = minhash(a), minhash(b)
estimate = sum(x == y for x, y in zip(ma, mb)) / len(ma)
print(f"Jaccard {jaccard:.2f}, MinHash estimate {estimate:.2f}")
# 5 -> 4 documents after exact dedup
# Jaccard 0.80, MinHash estimate 0.83


In [ ]:
# Packing: append <|endoftext|> to each document, concatenate, cut into blocks of n + 1 tokens
stream = []
for doc in unique:
    stream += enc.encode(doc) + [EOT]
n = 8
blocks = torch.tensor(stream[: len(stream) // (n + 1) * (n + 1)]).view(-1, n + 1)
inputs, labels = blocks[:, :-1], blocks[:, 1:]
print(len(stream), "tokens ->", tuple(blocks.shape), "blocks; dropped", len(stream) % (n + 1))
print([enc.decode([t]) for t in inputs[4].tolist()])
# 79 tokens -> (8, 9) blocks; dropped 7
# [' is', ' measured', ' in', ' tokens', '.', '<|endoftext|>', 'The', ' museum']
